# Chat with the LLMs

This notebook sets up a chat interface with a chosen model (base or finetuned) and provides the output with and without RAG.

## Prerequisites

The RAG implementation requires the vector database is pre-generated. The GitHub Actions workflow should keep it updated.

If not, run the [document generation script](retrieval_db_update.ipynb).

Using the fine-tuned model requires the chat model has been trained on the data.

If it hasn't been run yet, run the [model finetuning notebook](finetuning.ipynb).

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from retrieval import Retrieval

In [11]:
# Whether to apply the fine-tuned LoRA adapter to the weights.
LORA = True
# Whether to quantize the base model weights to reduce the memory footprint.
QUANTIZATION = True

# Chat LLM model name
CHAT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# Fine-tuned LoRA adapter directory
LORA_ADAPTER = f"/opt/shared/lora/{CHAT_MODEL}-Finetuned"

In [12]:
# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: NVIDIA A100 80GB PCIe


In [13]:
# Use default embeddings model, sentence-transformers/all-distilroberta-v1
# Use the default ChromaDB directory and collection
retrieval = Retrieval(log=True)

Loading the sentence transformer, sentence-transformers/all-distilroberta-v1 ...
Loading ChromaDB, /opt/shared/data/chromadb ...
Loading collection, all-documents ...
Initializing retrieval done


In [14]:
# Load the model and tokenizer

print(f"Loading model and tokenizer: {CHAT_MODEL}")

chat_tokenizer = AutoTokenizer.from_pretrained(
    CHAT_MODEL,
)
# Ensure padding token is set
if chat_tokenizer.pad_token is None:
    chat_tokenizer.pad_token = chat_tokenizer.eos_token

print("Tokenizer loaded")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL,
    quantization_config=bnb_config if QUANTIZATION else None,
    torch_dtype=torch.bfloat16 if not QUANTIZATION else None,
    trust_remote_code=True,
    device_map="auto"
)

print(base_model.hf_device_map)  # Shows which layers are on which device
print(base_model.get_memory_footprint() / 1024**3)  # Memory in GiB

if LORA:
    chat_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
else:
    chat_model = base_model

print("Model loaded")

print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB")

# Create the pipeline
pipe = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device_map="auto"
)

Loading model and tokenizer: Qwen/Qwen2.5-7B-Instruct
Tokenizer loaded


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

{'': 0}
5.06946873664856


Device set to use cuda:0


Model loaded
VRAM after model load: 11.12 GiB
Reserved: 13.36 GiB


In [19]:
import copy
# adds some nice features to `input`
import readline

# Maintain conversation history
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
]

# Maintain RAG conversation history
rag_conversation = copy.deepcopy(conversation)
rag_conversation[0]["content"] += "\n\nAdditional context documents may be included with queries. Integrate relevant information from these documents seamlessly into comprehensive responses. When documents are unhelpful or off-topic, rely on your existing knowledge. Maintain your normal level of detail and insight regardless of document quality."

# Reset the RAG document injection history
retrieval.reset()

while True:
    user_input = input("> ")

    if not user_input or user_input in ["quit", "q", "exit"]:
        break

    for rag, history in [(False, conversation), (True, rag_conversation)]:
        if rag:
            user_input = retrieval.augment(user_input, 3, log=True)
        
        # Add user message to conversation
        history.append({"role": "user", "content": user_input})

        # Pass the conversation directly to the pipeline
        outputs = pipe(
            history,
            max_new_tokens=1024,
            # To ensure repeatability, always pick the most-likely candidate for the next token.
            # To achieve this, turn sampling off (~= setting temperature to zero, but torch doesn't like that)
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )
    
        # Extract the assistant's response
        response = outputs[0]["generated_text"][-1]["content"]
    
        # Add assistant response to conversation
        history.append({"role": "assistant", "content": response})
        
        print()
        if rag:
            print("RAG+", end="")
        print("LLM>", response)
        print()

    break # TODO REMOVE

>  What is Qiskit?



LLM> Qiskit is an open-source software development framework for developing quantum applications, running them on prototype quantum hardware and simulators, and analyzing the results. It provides tools for writing, visualizing, and debugging quantum programs, as well as for running these programs on various backends including quantum computers from different providers and classical computers.

Key components of Qiskit include:

1. **Quantum Circuits**: A high-level abstraction for building quantum programs using a Python API. Users can define their quantum algorithms by manipulating and routing qubits on physical devices.

2. **Backend Adapters**: Interfaces to connect with different quantum computing platforms (e.g., IBM Quantum, Google Quantum AI, Rigetti, etc.) so that users can run their circuits on various types of quantum computers or simulators.

3. **Hardware Simulators**: Tools for running quantum circuits on classical computers to simulate the behavior of quantum systems. Th

In [ ]:
import gc

# Delete the GPU-hogging resources
del pipe
del chat_model
del base_model
del chat_tokenizer

# Force gc
gc.collect()

# Clear IPython's execution result cache
ip = get_ipython()
ip.displayhook.flush()

# Clear all In/Out history
ip.history_manager.reset(new_session=False)
ip.history_manager.input_hist_parsed[:] = []
ip.history_manager.input_hist_raw[:] = []
ip.history_manager.output_hist.clear()
ip.history_manager.output_hist_reprs.clear()
ip.history_manager.dir_hist[:] = []

# Clear the user namespace cache
ip.user_ns_hidden.clear()

# Force gc
gc.collect()

# More aggressive cache clearing
torch.cuda.empty_cache()
torch.cuda.synchronize()  # Wait for all operations to complete
torch.cuda.empty_cache()